In [ ]:
import pandas as pd 
    
    # Load time series data from CSV file
file_path = "/Users/igormol/Desktop/John/0105010006.csv"
data = pd.read_csv(file_path)

In [ ]:
# Monte Carlo Anomaly Detector and Treatment for John Deere's Project
#
# Igor Mol <igor.mol@makes.ai>
#
# The provided Python code implements a Monte Carlo approach for detecting and 
# substituting outliers in a time series dataset. Monte Carlo methods are statistical 
# techniques that use random sampling to obtain numerical results. In this context, 
# the code leverages the Z-score method, a statistical measure that quantifies how far
# away a particular data point is from the mean of a dataset. By calculating the 
# Z-scores for each data point in the "COUNT" column of the time series, the code 
# identifies outliers—those points that deviate significantly from the mean. If 
# the Z-score surpasses a predefined threshold, the corresponding data point is 
# considered an outlier and is replaced with NaN. This approach allows for the 
# identification and treatment of extreme values in the time series data, enhancing
# the robustness of subsequent analyses.
# 	The primary motivation for employing Monte Carlo methods in outlier detection 
# lies in their ability to handle complex and uncertain systems. Monte Carlo techniques
# use random sampling, allowing for the exploration of a wide range of possible scenarios.
# In the context of outlier detection, this flexibility is advantageous when dealing 
# with diverse datasets and complex patterns that may be challenging to capture using
# deterministic methods alone. The Z-score-based outlier detection mechanism in the 
# code provides a statistically sound approach to identify and address outliers in time
# series data, which is crucial for ensuring the accuracy and reliability of subsequent
# analyses and modeling efforts.
# 	The code not only implements the outlier detection technique but also includes
# functions for generating synthetic time series data, plotting the original and cleaned 
# data on the same graph for visual comparison, and displaying the original and cleaned 
# datasets in well-formatted tables. These additional features contribute to the pedagogical
# value of the code by providing a comprehensive example of how to apply Monte Carlo 
# methods for outlier detection in a real-world scenario, while also emphasizing the
# importance of data visualization and presentation for effective communication of results
# in data science.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Generate synthetic data
def generate_synthetic_data():
    np.random.seed(42)
    data = pd.DataFrame({
        'TIME_SCALE': np.arange(1, 14),
        'COUNT': np.random.randint(10, 50, size=13)
    })
    return data

# Function to detect and substitute outliers using Z-score
def monte_carlo_outlier_detection(data_frame, column, threshold=1):
    z_scores = np.abs((data_frame[column] - data_frame[column].mean()) / data_frame[column].std())
    outliers = z_scores > threshold
    data_frame.loc[outliers, column] = np.nan
    return data_frame

# Plot original and cleaned data on the same plot
def plot_original_vs_cleaned(original_data, cleaned_data):
    sns.set(style="whitegrid")
    plt.figure(figsize=(10, 6))

    # Plot original data
    sns.scatterplot(x='TIME_SCALE', y='COUNT', data=original_data, label='Original Data', color='skyblue')

    # Plot cleaned data
    sns.scatterplot(x='TIME_SCALE', y='COUNT', data=cleaned_data, label='Cleaned Data', color='salmon', marker='X')

    plt.title('Original vs. Cleaned Data')
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.legend()
    plt.show()

# Save the resulting time series in a data frame called "new_data"
def save_cleaned_data(original_data, column):
    new_data = original_data.copy()
    new_data = monte_carlo_outlier_detection(new_data, column)
    new_data.reset_index(drop=True, inplace=True)
    return new_data

# Display the data frames with improved formatting
def display_data_frames(original_data, cleaned_data):
    pd.set_option('display.precision', 2)
    
    print("\nOriginal Data:")
    print(original_data.to_string(index=False))

    print("\nCleaned Data:")
    print(cleaned_data.to_string(index=False))

# Main function to execute the steps
def main():
    original_data = generate_synthetic_data()
    cleaned_data = save_cleaned_data(original_data, 'COUNT')
    plot_original_vs_cleaned(original_data, cleaned_data)
    display_data_frames(original_data, cleaned_data)

if __name__ == "__main__":
    main()


In [ ]:
new_data = data

In [ ]:
# Sliding-window Approach to Time Series Forecasting Using Gaussiann SVM
#
# Igor Mol <igor.mol@makes.ai>
#
# The following code implements a Time Series Model using the Support Vector 
# Regression (SVM) technique for forecasting. The model is encapsulated within a
# class, facilitating organized and modular code structure. The overarching goal
# of this code is to perform time series analysis on a given dataset named 'data.'
# The dataset undergoes preprocessing, which involves sorting by the 'TIME_SCALE'
# column and handling missing values in the 'COUNT' column by dropping corresponding
# rows or filling NaN values with the mean. The time series modeling is conducted 
# through a sliding window approach, where historical predictions are generated 
# using SVM based on a specified window size. The extrapolation into the future
# is achieved by iteratively predicting subsequent time points, incorporating a
# randomness factor to introduce variability into the model's predictions. 
# The code also calculates performance metrics, such as Mean Absolute Error (MAE), 
# Covariance Matrix, and Standard Deviation, providing quantitative assessments
# of the model's accuracy.
#     The SVM model is constructed using the Scikit-Learn library, employing a
# radial basis function (RBF) kernel for flexibility in capturing complex 
# relationships within the time series. The sliding window predictions leverage
# historical data to iteratively train and predict, while the extrapolation step
# extends predictions into the future for a specified number of months. The
# randomness factor introduces a stochastic element, acknowledging the inherent
# uncertainty in future predictions.

import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=3, months_to_predict=12, randomness_factor=1):
        """
        Initialize the TimeSeriesModel with input data and configuration parameters.

        Parameters:
        - data: Input time-series data as a Pandas DataFrame.
        - window_size: Size of the sliding window for predictions.
        - months_to_predict: Number of months to extrapolate into the future.
        - randomness_factor: Factor to introduce randomness in future extrapolation.
        """
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  # List to store model predictions
        self.future_extrapolation = []  # List to store future extrapolation time points

    def preprocess_data(self):
        """
        Preprocess the input data by sorting, removing NaN values, and filling missing values.
        """
        # Sort the data based on the 'TIME_SCALE' column
        self.data = self.data.sort_values(by='TIME_SCALE')
        # Drop rows where 'COUNT' is NaN
        self.data = self.data.dropna(subset=['COUNT'])

        # Fill NaN values in 'COUNT' with the mean of the column
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        """
        Train a Support Vector Regression (SVR) model with an RBF kernel.

        Parameters:
        - X: Input features.
        - y: Target variable.

        Returns:
        Trained SVR model.
        """
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        """
        Generate predictions using a trained model.

        Parameters:
        - X: Input features for prediction.
        - model: Trained model.

        Returns:
        Array of predictions.
        """
        return model.predict(X)

    def sliding_window_predictions(self):
        """
        Perform sliding window predictions on the input time series data.
        """
        # Iterate through the data using a sliding window approach
        for i in range(len(self.data) - self.window_size + 1):
            # Extract a window of data
            window_data = self.data.iloc[i:i + self.window_size]
            # Prepare features (X) and target variable (y) for the model
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            # Skip if there are NaN values in the target variable
            if np.isnan(y).any():
                continue

            # Train the model using the current window
            model = self.train_model(X, y)

            # If there are more data points, predict the next time point and store the prediction
            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

                # Add 'PREDICTION' column to the DataFrame
                self.data.at[i + self.window_size, 'PREDICTION'] = prediction[0]

    def extrapolate_future(self):
        """
        Extrapolate future data points for a specified number of months.
        """
        # Iterate for the specified number of months
        for _ in range(self.months_to_predict):
            # Prepare features (X) and target variable (y) for the model using all available data
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            # Skip if there are NaN values in the target variable
            if np.isnan(y).any():
                continue

            # Train the model using all available data
            model = self.train_model(X, y)

            # Predict the next time point and add a random factor
            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))

    def calculate_metrics(self):
        """
        Calculate Mean Absolute Error (MAE), covariance matrix, and standard deviation.
        """
        mae = mean_absolute_error(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        covariance_matrix = np.cov(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        std_deviation = np.std(self.data['COUNT'][:len(self.data)] - self.predictions[:len(self.data)])
        return mae, covariance_matrix, std_deviation

    def plot_results(self):
        """
        Plot the original data, model predictions, and future extrapolation.
        """
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate((self.data['PREDICTION'], self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        """
        Display a table of calculated metrics.

        Parameters:
        - mae: Mean Absolute Error.
        - covariance_matrix: Covariance matrix.
        - std_deviation: Standard deviation.
        """
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))


def main():
    # Assuming your DataFrame is named 'data'
    # If not, replace 'data' with your actual DataFrame name
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    mae, covariance_matrix, std_deviation = time_series_model.calculate_metrics()
    time_series_model.plot_results()
    time_series_model.display_results_table(mae, covariance_matrix, std_deviation)


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Function to detect and substitute outliers using Bollinger Bands
def bollinger_band_outlier_detection(data_frame, column, window_size=1, num_std=2):
    data_frame['ROLLING_MEAN'] = data_frame[column].rolling(window=window_size).mean()
    data_frame['UPPER_BAND'] = data_frame['ROLLING_MEAN'] + num_std * data_frame[column].rolling(window=window_size).std()
    data_frame['LOWER_BAND'] = data_frame['ROLLING_MEAN'] - num_std * data_frame[column].rolling(window=window_size).std()

    outliers = (data_frame[column] > data_frame['UPPER_BAND']) | (data_frame[column] < data_frame['LOWER_BAND'])
    data_frame.loc[outliers, column] = np.nan

    data_frame.drop(['ROLLING_MEAN', 'UPPER_BAND', 'LOWER_BAND'], axis=1, inplace=True)

    return data_frame

# Plot original and cleaned data
def plot_original_vs_cleaned(original_data, cleaned_data):
    plt.figure(figsize=(10, 6))
    plt.plot(original_data['TIME_SCALE'], original_data['COUNT'], label='Original Data', marker='o')
    plt.plot(cleaned_data['TIME_SCALE'], cleaned_data['COUNT'], label='Cleaned Data', marker='X')
    plt.title('Original vs. Cleaned Data')
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.legend()
    plt.show()

# Save the resulting time series in a data frame called "new_data"
def save_cleaned_data(original_data, column):
    new_data = original_data.copy()
    new_data = bollinger_band_outlier_detection(new_data, column)
    new_data.reset_index(drop=True, inplace=True)
    return new_data

# Display the original and cleaned data
original_data = data.copy()
cleaned_data = save_cleaned_data(original_data, 'COUNT')
plot_original_vs_cleaned(original_data, cleaned_data)

# Save the cleaned data in a new_data data frame
new_data = cleaned_data.copy()

# Display the resulting time series
print("Original Data:")
print(original_data)
print("\nCleaned Data:")
print(new_data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Function to detect and substitute outliers using Z-score
def z_score_outlier_detection(data_frame, column, threshold=1):
    z_scores = np.abs((data_frame[column] - data_frame[column].mean()) / data_frame[column].std())
    outliers = z_scores > threshold
    data_frame.loc[outliers, column] = np.nan
    return data_frame

# Plot original and cleaned data
def plot_original_vs_cleaned(original_data, cleaned_data):
    plt.figure(figsize=(10, 6))
    plt.plot(original_data['TIME_SCALE'], original_data['COUNT'], label='Original Data', marker='o')
    plt.plot(cleaned_data['TIME_SCALE'], cleaned_data['COUNT'], label='Cleaned Data', marker='X')
    plt.title('Original vs. Cleaned Data')
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.legend()
    plt.show()

# Save the resulting time series in a data frame called "new_data"
def save_cleaned_data(original_data, column):
    new_data = original_data.copy()
    new_data = z_score_outlier_detection(new_data, column)
    new_data.reset_index(drop=True, inplace=True)
    return new_data

# Display the original and cleaned data
original_data = data.copy()
cleaned_data = save_cleaned_data(original_data, 'COUNT')
plot_original_vs_cleaned(original_data, cleaned_data)

# Save the cleaned data in a new_data data frame
new_data = cleaned_data.copy()

# Display the resulting time series
print("Original Data:")
print(original_data)
print("\nCleaned Data:")
print(new_data)


In [ ]:
# Sliding-window Approach to Time Series Forecasting Using Gaussiann SVM
#
# Igor Mol <igor.mol@makes.ai>
#
# The following code implements a Time Series Model using the Support Vector 
# Regression (SVM) technique for forecasting. The model is encapsulated within a
# class, facilitating organized and modular code structure. The overarching goal
# of this code is to perform time series analysis on a given dataset named 'data.'
# The dataset undergoes preprocessing, which involves sorting by the 'TIME_SCALE'
# column and handling missing values in the 'COUNT' column by dropping corresponding
# rows or filling NaN values with the mean. The time series modeling is conducted 
# through a sliding window approach, where historical predictions are generated 
# using SVM based on a specified window size. The extrapolation into the future
# is achieved by iteratively predicting subsequent time points, incorporating a
# randomness factor to introduce variability into the model's predictions. 
# The code also calculates performance metrics, such as Mean Absolute Error (MAE), 
# Covariance Matrix, and Standard Deviation, providing quantitative assessments
# of the model's accuracy.
#     The SVM model is constructed using the Scikit-Learn library, employing a
# radial basis function (RBF) kernel for flexibility in capturing complex 
# relationships within the time series. The sliding window predictions leverage
# historical data to iteratively train and predict, while the extrapolation step
# extends predictions into the future for a specified number of months. The
# randomness factor introduces a stochastic element, acknowledging the inherent
# uncertainty in future predictions.

import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=3, months_to_predict=12, randomness_factor=1):
        """
        Initialize the TimeSeriesModel with input data and configuration parameters.

        Parameters:
        - data: Input time-series data as a Pandas DataFrame.
        - window_size: Size of the sliding window for predictions.
        - months_to_predict: Number of months to extrapolate into the future.
        - randomness_factor: Factor to introduce randomness in future extrapolation.
        """
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  # List to store model predictions
        self.future_extrapolation = []  # List to store future extrapolation time points

    def preprocess_data(self):
        """
        Preprocess the input data by sorting, removing NaN values, and filling missing values.
        """
        # Sort the data based on the 'TIME_SCALE' column
        self.data = self.data.sort_values(by='TIME_SCALE')
        # Drop rows where 'COUNT' is NaN
        self.data = self.data.dropna(subset=['COUNT'])

        # Fill NaN values in 'COUNT' with the mean of the column
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        """
        Train a Support Vector Regression (SVR) model with an RBF kernel.

        Parameters:
        - X: Input features.
        - y: Target variable.

        Returns:
        Trained SVR model.
        """
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        """
        Generate predictions using a trained model.

        Parameters:
        - X: Input features for prediction.
        - model: Trained model.

        Returns:
        Array of predictions.
        """
        return model.predict(X)

    def sliding_window_predictions(self):
        """
        Perform sliding window predictions on the input time series data.
        """
        # Iterate through the data using a sliding window approach
        for i in range(len(self.data) - self.window_size + 1):
            # Extract a window of data
            window_data = self.data.iloc[i:i + self.window_size]
            # Prepare features (X) and target variable (y) for the model
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            # Skip if there are NaN values in the target variable
            if np.isnan(y).any():
                continue

            # Train the model using the current window
            model = self.train_model(X, y)

            # If there are more data points, predict the next time point and store the prediction
            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

                # Add 'PREDICTION' column to the DataFrame
                self.data.at[i + self.window_size, 'PREDICTION'] = prediction[0]

    def extrapolate_future(self):
        """
        Extrapolate future data points for a specified number of months.
        """
        # Iterate for the specified number of months
        for _ in range(self.months_to_predict):
            # Prepare features (X) and target variable (y) for the model using all available data
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            # Skip if there are NaN values in the target variable
            if np.isnan(y).any():
                continue

            # Train the model using all available data
            model = self.train_model(X, y)

            # Predict the next time point and add a random factor
            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))

    def calculate_metrics(self):
        """
        Calculate Mean Absolute Error (MAE), covariance matrix, and standard deviation.
        """
        mae = mean_absolute_error(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        covariance_matrix = np.cov(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        std_deviation = np.std(self.data['COUNT'][:len(self.data)] - self.predictions[:len(self.data)])
        return mae, covariance_matrix, std_deviation

    def plot_results(self):
        """
        Plot the original data, model predictions, and future extrapolation.
        """
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate((self.data['PREDICTION'], self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        """
        Display a table of calculated metrics.

        Parameters:
        - mae: Mean Absolute Error.
        - covariance_matrix: Covariance matrix.
        - std_deviation: Standard deviation.
        """
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))


def main():
    # Assuming your DataFrame is named 'data'
    # If not, replace 'data' with your actual DataFrame name
    data = new_data
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    mae, covariance_matrix, std_deviation = time_series_model.calculate_metrics()
    time_series_model.plot_results()
    time_series_model.display_results_table(mae, covariance_matrix, std_deviation)


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=3, months_to_predict=12, randomness_factor=1):
        """
        Initialize the TimeSeriesModel with input data and configuration parameters.

        Parameters:
        - data: Input time-series data as a Pandas DataFrame.
        - window_size: Size of the sliding window for predictions.
        - months_to_predict: Number of months to extrapolate into the future.
        - randomness_factor: Factor to introduce randomness in future extrapolation.
        """
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  # List to store model predictions
        self.future_extrapolation = []  # List to store future extrapolation time points

    def preprocess_data(self):
        """
        Preprocess the input data by sorting, removing NaN values, and filling missing values.
        """
        # Sort the data based on the 'TIME_SCALE' column
        self.data = self.data.sort_values(by='TIME_SCALE')
        # Drop rows where 'COUNT' is NaN
        self.data = self.data.dropna(subset=['COUNT'])

        # Fill NaN values in 'COUNT' with the mean of the column
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        """
        Train a Support Vector Regression (SVR) model with an RBF kernel.

        Parameters:
        - X: Input features.
        - y: Target variable.

        Returns:
        Trained SVR model.
        """
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        """
        Generate predictions using a trained model.

        Parameters:
        - X: Input features for prediction.
        - model: Trained model.

        Returns:
        Array of predictions.
        """
        # Check for NaN values in the input features
        if np.isnan(X).any():
            return np.full(len(X), np.nan)

        return model.predict(X)

    def sliding_window_predictions(self):
        """
        Perform sliding window predictions on the input time series data.
        """
        # Iterate through the data using a sliding window approach
        for i in range(len(self.data) - self.window_size + 1):
            # Extract a window of data
            window_data = self.data.iloc[i:i + self.window_size]
            # Prepare features (X) and target variable (y) for the model
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            # Skip if there are NaN values in the target variable
            if np.isnan(y).any():
                continue

            # Train the model using the current window
            model = self.train_model(X, y)

            # If there are more data points, predict the next time point and store the prediction
            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

                # Add 'PREDICTION' column to the DataFrame
                self.data.at[i + self.window_size, 'PREDICTION'] = prediction[0]

    def extrapolate_future(self):
        """
        Extrapolate future data points for a specified number of months.
        """
        # Iterate for the specified number of months
        for _ in range(self.months_to_predict):
            # Prepare features (X) and target variable (y) for the model using all available data
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            # Skip if there are NaN values in the target variable
            if np.isnan(y).any():
                continue

            # Train the model using all available data
            model = self.train_model(X, y)

            # Predict the next time point and add a random factor
            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))

    def calculate_metrics(self):
        """
        Calculate Mean Absolute Error (MAE), covariance matrix, and standard deviation.
        """
        mae = mean_absolute_error(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        covariance_matrix = np.cov(self.data['COUNT'][:len(self.data)], self.predictions[:len(self.data)])
        std_deviation = np.std(self.data['COUNT'][:len(self.data)] - self.predictions[:len(self.data)])
        return mae, covariance_matrix, std_deviation

    def plot_results(self):
        """
        Plot the original data, model predictions, and future extrapolation.
        """
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate((self.data['PREDICTION'], self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        """
        Display a table of calculated metrics.

        Parameters:
        - mae: Mean Absolute Error.
        - covariance_matrix: Covariance matrix.
        - std_deviation: Standard deviation.
        """
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))


def main():
    # Assuming your DataFrame is named 'data'
    # If not, replace 'data' with your actual DataFrame name
    data = new_data
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    mae, covariance_matrix, std_deviation = time_series_model.calculate_metrics()
    time_series_model.plot_results()
    time_series_model.display_results_table(mae, covariance_matrix, std_deviation)


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=3, months_to_predict=12, randomness_factor=1):
        """
        Initialize the TimeSeriesModel with input data and configuration parameters.

        Parameters:
        - data: Input time-series data as a Pandas DataFrame.
        - window_size: Size of the sliding window for predictions.
        - months_to_predict: Number of months to extrapolate into the future.
        - randomness_factor: Factor to introduce randomness in future extrapolation.
        """
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  # List to store model predictions
        self.future_extrapolation = []  # List to store future extrapolation time points

    def preprocess_data(self):
        """
        Preprocess the input data by sorting, removing NaN values, and filling missing values.
        """
        # Sort the data based on the 'TIME_SCALE' column
        self.data = self.data.sort_values(by='TIME_SCALE')
        # Drop rows where 'COUNT' is NaN
        self.data = self.data.dropna(subset=['COUNT'])

        # Fill NaN values in 'COUNT' with the mean of the column
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        """
        Train a Support Vector Regression (SVR) model with an RBF kernel.

        Parameters:
        - X: Input features.
        - y: Target variable.

        Returns:
        Trained SVR model.
        """
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        """
        Generate predictions using a trained model.

        Parameters:
        - X: Input features for prediction.
        - model: Trained model.

        Returns:
        Array of predictions.
        """
        # Check for NaN values in the input features
        if np.isnan(X).any():
            return np.full(len(X), np.nan)

        return model.predict(X)

    def sliding_window_predictions(self):
        """
        Perform sliding window predictions on the input time series data.
        """
        # Iterate through the data using a sliding window approach
        for i in range(len(self.data) - self.window_size + 1):
            # Extract a window of data
            window_data = self.data.iloc[i:i + self.window_size]
            # Prepare features (X) and target variable (y) for the model
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            # Skip if there are NaN values in the target variable
            if np.isnan(y).any():
                continue

            # Train the model using the current window
            model = self.train_model(X, y)

            # If there are more data points, predict the next time point and store the prediction
            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

                # Add 'PREDICTION' column to the DataFrame
                self.data.at[i + self.window_size, 'PREDICTION'] = prediction[0]

    def extrapolate_future(self):
        """
        Extrapolate future data points for a specified number of months.
        """
        # Iterate for the specified number of months
        for _ in range(self.months_to_predict):
            # Prepare features (X) and target variable (y) for the model using all available data
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            # Skip if there are NaN values in the target variable
            if np.isnan(y).any():
                continue

            # Train the model using all available data
            model = self.train_model(X, y)

            # Predict the next time point and add a random factor
            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))

    def calculate_metrics(self):
        """
        Calculate Mean Absolute Error (MAE), covariance matrix, and standard deviation.
        """
        # Remove NaN values from predictions and corresponding true values
        non_nan_indices = ~np.isnan(self.predictions)
        y_true = self.data['COUNT'][:len(self.data)][non_nan_indices]
        y_pred = np.array(self.predictions[:len(self.data)])[non_nan_indices]

        mae = mean_absolute_error(y_true, y_pred)
        covariance_matrix = np.cov(y_true, y_pred)
        std_deviation = np.std(y_true - y_pred)

        return mae, covariance_matrix, std_deviation

    def plot_results(self):
        """
        Plot the original data, model predictions, and future extrapolation.
        """
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate((self.data['PREDICTION'], self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        """
        Display a table of calculated metrics.

        Parameters:
        - mae: Mean Absolute Error.
        - covariance_matrix: Covariance matrix.
        - std_deviation: Standard deviation.
        """
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))


def main():
    # Assuming your DataFrame is named 'data'
    # If not, replace 'data' with your actual DataFrame name
    data = new_data
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    mae, covariance_matrix, std_deviation = time_series_model.calculate_metrics()
    time_series_model.plot_results()
    time_series_model.display_results_table(mae, covariance_matrix, std_deviation)


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=3, months_to_predict=12, randomness_factor=1):
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []
        self.future_extrapolation = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        return model.predict(X)

    def sliding_window_predictions(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        for _ in range(self.months_to_predict):
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))

def calculate_metrics(self):
    """
    Calculate Mean Absolute Error (MAE), covariance matrix, and standard deviation.
    """
    non_nan_indices = ~np.isnan(self.predictions)

    # Adjust indexing to ensure consistent lengths
    start_index = self.window_size
    end_index = start_index + len(self.predictions)
    y_true = self.data['COUNT'].iloc[start_index:end_index][non_nan_indices]
    y_pred = np.array(self.predictions)[non_nan_indices]

    mae = mean_absolute_error(y_true, y_pred)
    covariance_matrix = np.cov(y_true, y_pred)
    std_deviation = np.std(y_true - y_pred)

    return mae, covariance_matrix, std_deviation


    def plot_results(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate((self.data['PREDICTION'], self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))


def main():
    data = new_data  # Replace with your actual DataFrame
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    mae, covariance_matrix, std_deviation = time_series_model.calculate_metrics()
    time_series_model.plot_results()
    time_series_model.display_results_table(mae, covariance_matrix, std_deviation)


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=3, months_to_predict=12, randomness_factor=1):
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []
        self.future_extrapolation = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        return model.predict(X)

    def sliding_window_predictions(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        for _ in range(self.months_to_predict):
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))

    def calculate_metrics(self):
        non_nan_indices = ~np.isnan(self.predictions)

        # Adjust indexing to ensure consistent lengths
        start_index = self.window_size
        end_index = start_index + len(self.predictions)
        y_true = self.data['COUNT'].iloc[start_index:end_index][non_nan_indices]
        y_pred = np.array(self.predictions)[non_nan_indices]

        mae = mean_absolute_error(y_true, y_pred)
        covariance_matrix = np.cov(y_true, y_pred)
        std_deviation = np.std(y_true - y_pred)

        return mae, covariance_matrix, std_deviation

    def plot_results(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate((self.data['PREDICTION'], self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))


def main():
    data = new_data  # Replace with your actual DataFrame
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    mae, covariance_matrix, std_deviation = time_series_model.calculate_metrics()
    time_series_model.plot_results()
    time_series_model.display_results_table(mae, covariance_matrix, std_deviation)


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=3, months_to_predict=12, randomness_factor=1):
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []
        self.future_extrapolation = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        return model.predict(X)

    def sliding_window_predictions(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        for _ in range(self.months_to_predict):
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))

    def calculate_metrics(self):
        start_index = self.window_size
        end_index = start_index + min(len(self.predictions), len(self.data) - self.window_size)

        y_true = self.data['COUNT'].iloc[start_index:end_index]
        y_pred = np.array(self.predictions)[:end_index - start_index]

        mae = mean_absolute_error(y_true, y_pred)
        covariance_matrix = np.cov(y_true, y_pred)
        std_deviation = np.std(y_true - y_pred)

        return mae, covariance_matrix, std_deviation

    def plot_results(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate((self.data['PREDICTION'], self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))


def main():
    data = new_data  # Replace with your actual DataFrame
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    mae, covariance_matrix, std_deviation = time_series_model.calculate_metrics()
    time_series_model.plot_results()
    time_series_model.display_results_table(mae, covariance_matrix, std_deviation)


if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=1, months_to_predict=12, randomness_factor=1):
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  
        self.future_extrapolation = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        return model.predict(X)

    def sliding_window_predictions(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        for _ in range(self.months_to_predict):
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))


    def plot_results(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate(([np.nan] * len(self.data['TIME_SCALE']), self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))

def main():
    data = new_data  # Replace 'new_data' with your actual DataFrame
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    time_series_model.plot_results()

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Function to detect and substitute outliers using Z-score
def z_score_outlier_detection(data_frame, column, threshold=1):
    z_scores = np.abs((data_frame[column] - data_frame[column].mean()) / data_frame[column].std())
    outliers = z_scores > threshold
    data_frame.loc[outliers, column] = np.nan
    return data_frame

# Plot original and cleaned data
def plot_original_vs_cleaned(original_data, cleaned_data):
    plt.figure(figsize=(10, 6))
    plt.plot(original_data['TIME_SCALE'], original_data['COUNT'], label='Original Data', marker='o')
    plt.plot(cleaned_data['TIME_SCALE'], cleaned_data['COUNT'], label='Cleaned Data', marker='X')
    plt.title('Original vs. Cleaned Data')
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.legend()
    plt.show()

# Save the resulting time series in a data frame called "new_data"
def save_cleaned_data(original_data, column):
    new_data = original_data.copy()
    new_data = z_score_outlier_detection(new_data, column)
    new_data.reset_index(drop=True, inplace=True)
    return new_data

# Display the original and cleaned data
original_data = data.copy()
cleaned_data = save_cleaned_data(original_data, 'COUNT')
plot_original_vs_cleaned(original_data, cleaned_data)

# Save the cleaned data in a new_data data frame
new_data = cleaned_data.copy()

# Display the resulting time series
print("Original Data:")
print(original_data)
print("\nCleaned Data:")
print(new_data)



In [ ]:
data = new_data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=10, months_to_predict=12, randomness_factor=1):
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  
        self.future_extrapolation = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        return model.predict(X)

    def sliding_window_predictions(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        for _ in range(self.months_to_predict):
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))


    def plot_results(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate(([np.nan] * len(self.data['TIME_SCALE']), self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))

def main():
    data = new_data  # Replace 'new_data' with your actual DataFrame
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    time_series_model.plot_results()

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=1, months_to_predict=12, randomness_factor=1):
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  
        self.future_extrapolation = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        return model.predict(X)

    def sliding_window_predictions(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        for _ in range(self.months_to_predict):
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))

    def calculate_metrics(self):
        start_index = self.window_size
        end_index = start_index + len(self.predictions)
        y_true = self.data['COUNT'].iloc[start_index:end_index].values
        non_nan_indices = ~np.isnan(self.predictions)
        y_pred = np.array(self.predictions)[non_nan_indices]
        mae = mean_absolute_error(y_true, y_pred)
        covariance_matrix = np.cov(y_true, y_pred)
        std_deviation = np.std(y_true - y_pred)
        return mae, covariance_matrix, std_deviation

    def plot_results(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate(([np.nan] * len(self.data['TIME_SCALE']), self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))

def main():
    # Assuming 'data' is the original DataFrame
    original_data = data.copy()
    
    # Data cleaning
    cleaned_data = z_score_outlier_detection(original_data, 'COUNT')
    
    # Time series modeling
    time_series_model = TimeSeriesModel(cleaned_data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    mae, covariance_matrix, std_deviation = time_series_model.calculate_metrics()
    time_series_model.plot_results()
    time_series_model.display_results_table(mae, covariance_matrix, std_deviation)

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=10, months_to_predict=12, randomness_factor=0.8):
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  
        self.future_extrapolation = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        return model.predict(X)

    def sliding_window_predictions(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        for _ in range(self.months_to_predict):
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))


    def plot_results(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate(([np.nan] * len(self.data['TIME_SCALE']), self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))

def main():
    data = new_data  # Replace 'new_data' with your actual DataFrame
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    time_series_model.plot_results()

if __name__ == "__main__":
    main()


In [ ]:
data.info()

In [ ]:
data

In [ ]:
import pandas as pd

# Assuming your original data frame is named "data"

# Create a new_data data frame with TIME_SCALE ranging from 1 to 135
new_data = pd.DataFrame({'TIME_SCALE': range(1, 136)})

# Merge new_data with the original data on TIME_SCALE using left join
new_data = new_data.merge(data, on='TIME_SCALE', how='left')

# Fill NaN values in COUNT column with 0
new_data['COUNT'].fillna(0, inplace=True)

# If you want to ensure COUNT column is of integer type, you can convert it
new_data['COUNT'] = new_data['COUNT'].astype(int)

# Display the resulting new_data data frame
print(new_data)


In [ ]:
data = new_data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=3, months_to_predict=12, randomness_factor=0):
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  
        self.future_extrapolation = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        return model.predict(X)

    def sliding_window_predictions(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        for _ in range(self.months_to_predict):
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))


    def plot_results(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate(([np.nan] * len(self.data['TIME_SCALE']), self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))

def main():
    data = new_data  # Replace 'new_data' with your actual DataFrame
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    time_series_model.plot_results()

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Function to detect and substitute outliers using Z-score
def z_score_outlier_detection(data_frame, column, threshold=1):
    z_scores = np.abs((data_frame[column] - data_frame[column].mean()) / data_frame[column].std())
    outliers = z_scores > threshold
    data_frame.loc[outliers, column] = np.nan
    return data_frame

# Plot original and cleaned data
def plot_original_vs_cleaned(original_data, cleaned_data):
    plt.figure(figsize=(10, 6))
    plt.plot(original_data['TIME_SCALE'], original_data['COUNT'], label='Original Data', marker='o')
    plt.plot(cleaned_data['TIME_SCALE'], cleaned_data['COUNT'], label='Cleaned Data', marker='X')
    plt.title('Original vs. Cleaned Data')
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.legend()
    plt.show()

# Save the resulting time series in a data frame called "new_data"
def save_cleaned_data(original_data, column):
    new_data = original_data.copy()
    new_data = z_score_outlier_detection(new_data, column)
    new_data.reset_index(drop=True, inplace=True)
    return new_data

# Display the original and cleaned data
original_data = data.copy()
cleaned_data = save_cleaned_data(original_data, 'COUNT')
plot_original_vs_cleaned(original_data, cleaned_data)

# Save the cleaned data in a new_data data frame
new_data = cleaned_data.copy()

# Display the resulting time series
print("Original Data:")
print(original_data)
print("\nCleaned Data:")
print(new_data)



In [ ]:
data = new_data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=1, months_to_predict=12, randomness_factor=0.1):
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  
        self.future_extrapolation = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        return model.predict(X)

    def sliding_window_predictions(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        for _ in range(self.months_to_predict):
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))


    def plot_results(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate(([np.nan] * len(self.data['TIME_SCALE']), self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))

def main():
    data = new_data  # Replace 'new_data' with your actual DataFrame
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    time_series_model.plot_results()

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Function to detect and substitute outliers using Z-score
def z_score_outlier_detection(data_frame, column, threshold=2):
    z_scores = np.abs((data_frame[column] - data_frame[column].mean()) / data_frame[column].std())
    outliers = z_scores > threshold
    data_frame.loc[outliers, column] = np.nan
    return data_frame

# Plot original and cleaned data
def plot_original_vs_cleaned(original_data, cleaned_data):
    plt.figure(figsize=(10, 6))
    plt.plot(original_data['TIME_SCALE'], original_data['COUNT'], label='Original Data', marker='o')
    plt.plot(cleaned_data['TIME_SCALE'], cleaned_data['COUNT'], label='Cleaned Data', marker='X')
    plt.title('Original vs. Cleaned Data')
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.legend()
    plt.show()

# Save the resulting time series in a data frame called "new_data"
def save_cleaned_data(original_data, column):
    new_data = original_data.copy()
    new_data = z_score_outlier_detection(new_data, column)
    new_data.reset_index(drop=True, inplace=True)
    return new_data

# Display the original and cleaned data
original_data = data.copy()
cleaned_data = save_cleaned_data(original_data, 'COUNT')
plot_original_vs_cleaned(original_data, cleaned_data)

# Save the cleaned data in a new_data data frame
new_data = cleaned_data.copy()

# Display the resulting time series
print("Original Data:")
print(original_data)
print("\nCleaned Data:")
print(new_data)



In [ ]:
import pandas as pd

# Assuming your original data frame is named "data"

# Create a new_data data frame with TIME_SCALE ranging from 1 to 135
new_data = pd.DataFrame({'TIME_SCALE': range(1, 136)})

# Merge new_data with the original data on TIME_SCALE using left join
new_data = new_data.merge(data, on='TIME_SCALE', how='left')

# Fill NaN values in COUNT column with 0
new_data['COUNT'].fillna(0, inplace=True)

# If you want to ensure COUNT column is of integer type, you can convert it
new_data['COUNT'] = new_data['COUNT'].astype(int)

# Display the resulting new_data data frame
print(new_data)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Function to detect and substitute outliers using Z-score
def z_score_outlier_detection(data_frame, column, threshold=2.5):
    z_scores = np.abs((data_frame[column] - data_frame[column].mean()) / data_frame[column].std())
    outliers = z_scores > threshold
    data_frame.loc[outliers, column] = np.nan
    return data_frame

# Plot original and cleaned data
def plot_original_vs_cleaned(original_data, cleaned_data):
    plt.figure(figsize=(10, 6))
    plt.plot(original_data['TIME_SCALE'], original_data['COUNT'], label='Original Data', marker='o')
    plt.plot(cleaned_data['TIME_SCALE'], cleaned_data['COUNT'], label='Cleaned Data', marker='X')
    plt.title('Original vs. Cleaned Data')
    plt.xlabel('Time Scale')
    plt.ylabel('Count')
    plt.legend()
    plt.show()

# Save the resulting time series in a data frame called "new_data"
def save_cleaned_data(original_data, column):
    new_data = original_data.copy()
    new_data = z_score_outlier_detection(new_data, column)
    new_data.reset_index(drop=True, inplace=True)
    return new_data

# Display the original and cleaned data
original_data = data.copy()
cleaned_data = save_cleaned_data(original_data, 'COUNT')
plot_original_vs_cleaned(original_data, cleaned_data)

# Save the cleaned data in a new_data data frame
new_data = cleaned_data.copy()

# Display the resulting time series
print("Original Data:")
print(original_data)
print("\nCleaned Data:")
print(new_data)

In [ ]:
data = new_data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=3, months_to_predict=12, randomness_factor=0.1):
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  
        self.future_extrapolation = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        return model.predict(X)

    def sliding_window_predictions(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        for _ in range(self.months_to_predict):
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))


    def plot_results(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate(([np.nan] * len(self.data['TIME_SCALE']), self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))

def main():
    data = new_data  # Replace 'new_data' with your actual DataFrame
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    time_series_model.plot_results()

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
import seaborn as sns
import matplotlib.pyplot as plt
from tabulate import tabulate

class TimeSeriesModel:
    def __init__(self, data, window_size=1, months_to_predict=12, randomness_factor=0.5):
        self.data = data
        self.window_size = window_size
        self.months_to_predict = months_to_predict
        self.randomness_factor = randomness_factor
        self.predictions = []  
        self.future_extrapolation = []

    def preprocess_data(self):
        self.data = self.data.sort_values(by='TIME_SCALE')
        self.data = self.data.dropna(subset=['COUNT'])
        if self.data['COUNT'].isnull().sum() > 0:
            self.data['COUNT'].fillna(self.data['COUNT'].mean(), inplace=True)

    def train_model(self, X, y):
        model = SVR(kernel='rbf', gamma='scale')
        model.fit(X, y)
        return model

    def generate_predictions(self, X, model):
        return model.predict(X)

    def sliding_window_predictions(self):
        for i in range(len(self.data) - self.window_size + 1):
            window_data = self.data.iloc[i:i + self.window_size]
            X = window_data['TIME_SCALE'].values.reshape(-1, 1)
            y = window_data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            if i + self.window_size < len(self.data):
                next_time_point = self.data.iloc[i + self.window_size]['TIME_SCALE']
                self.future_extrapolation.append(next_time_point)
                prediction = self.generate_predictions([[next_time_point]], model)
                self.predictions.append(prediction[0])

    def extrapolate_future(self):
        for _ in range(self.months_to_predict):
            X = self.data['TIME_SCALE'].values.reshape(-1, 1)
            y = self.data['COUNT'].values

            if np.isnan(y).any():
                continue

            model = self.train_model(X, y)

            next_time_point = self.future_extrapolation[-1] + 1
            self.future_extrapolation.append(next_time_point)
            prediction = self.generate_predictions([[next_time_point]], model)
            self.predictions.append(prediction[0] + np.random.normal(scale=self.randomness_factor))


    def plot_results(self):
        sns.set(style="whitegrid")
        plt.figure(figsize=(12, 6))

        combined_time = np.concatenate((self.data['TIME_SCALE'], self.future_extrapolation))
        combined_predictions = np.concatenate(([np.nan] * len(self.data['TIME_SCALE']), self.predictions))

        sns.lineplot(x=self.data['TIME_SCALE'], y=self.data['COUNT'], label='Original Data', color='blue')
        sns.lineplot(x=combined_time, y=combined_predictions,
                     label='Model Prediction and Future Extrapolation', linestyle='dashed', color='green', marker='o')

        plt.xlabel('Time Scale')
        plt.ylabel('Count')
        plt.title(f'Time Series Modeling with Sliding Window and {self.months_to_predict} Months Future Extrapolation (SVM)')
        plt.legend()
        plt.show()

    def display_results_table(self, mae, covariance_matrix, std_deviation):
        results_table = pd.DataFrame({
            'Metric': ['Mean Absolute Error (MAE)', 'Covariance Matrix', 'Standard Deviation'],
            'Value': [mae, covariance_matrix, std_deviation]
        })

        print(tabulate(results_table, headers='keys', tablefmt='fancy_grid'))

def main():
    data = new_data  # Replace 'new_data' with your actual DataFrame
    time_series_model = TimeSeriesModel(data)
    time_series_model.preprocess_data()
    time_series_model.sliding_window_predictions()
    time_series_model.extrapolate_future()
    time_series_model.plot_results()

if __name__ == "__main__":
    main()
